# 📚 Notebook 1: GSEB Curriculum Ingestion & Offline OCR
**EduNavika** -- Easy Interactive Guide to Textbook Scanning & Text Extraction.

This notebook shows you how EduNavika discovers official GSEB textbook PDFs, cleans text, decodes font glyphs, and extracts Table of Contents (TOC) structure.


### Step 1: Discover and Hash Official GSEB Textbook PDFs
Let's scan the `GSEB-Dataset/` directory and calculate SHA-256 hashes for document integrity.


In [ ]:
import os
import hashlib

dataset_dir = 'GSEB-Dataset'
pdf_files = []

for root, _, files in os.walk(dataset_dir):
    for f in files:
        if f.endswith('.pdf'):
            rel_path = os.path.relpath(os.path.join(root, f), dataset_dir)
            pdf_files.append(rel_path)

print(f'Total GSEB Textbook PDFs Discovered: {len(pdf_files)}')
for pdf in pdf_files[:5]:
    print(f' - {pdf}')


### Step 2: Extract Text Page-by-Page using PyPDFium2
EduNavika extracts text deterministically while preserving exact physical 1-indexed page numbers.


In [ ]:
import pypdfium2 as pdfium

# Pick the first discovered PDF
if pdf_files:
    sample_pdf_path = os.path.join(dataset_dir, pdf_files[0])
    doc = pdfium.PdfDocument(sample_pdf_path)
    print(f'Inspecting: {pdf_files[0]}')
    print(f'Total Physical Pages: {len(doc)}')
    
    # Read page 1 text
    page_1 = doc[0]
    textpage = page_1.get_textpage()
    sample_text = textpage.get_text_range()
    print('\n--- Sample Text Preview (Page 1) ---')
    print(sample_text[:300] + '...')


### Step 3: Text Cleaning and Font Glyph Normalization
Raw PDF text often has broken typography and custom font glyphs like `/G<n>`. Here is how the cleaner normalizes it:


In [ ]:
from backend.app.ingestion.cleaner import clean_text

raw_sample = 'Chapter  1 :  Real   Numbers \n\nPage - 1\nSome text with  extra   spaces.'
cleaned = clean_text(raw_sample)
print('Original:', repr(raw_sample))
print('Cleaned: ', repr(cleaned))


### Step 4: Local Offline OCR Engine Demo
EduNavika uses `RapidOCR` with ONNX Runtime on CPU. Zero cloud API calls and zero cost.


In [ ]:
from backend.app.ingestion.ocr.ocr_engine import RapidOCREngine

ocr = RapidOCREngine()
print(f'OCR Engine Initialized: {ocr.name}')
print('Ready to transcribe scanned textbook pages completely offline on CPU.')
